In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_recall_fscore_support, average_precision_score

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torchvision
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import box_iou

# Custom dataset for object detection
class PlantRCNNDataset(Dataset):
    def __init__(self, df, class_map, transform=None, is_test=False):
        """
        Custom dataset for plant object detection with RCNN.
        
        Args:
            df: DataFrame with image paths and class info, including bounding box coordinates
            class_map: Dictionary mapping class names to indices
            transform: Image transformations to apply
            is_test: Whether this is a test dataset (no labels)
        """
        self.df = df
        self.class_map = class_map
        self.transform = transform
        self.is_test = is_test
        self.num_classes = len(class_map) + 1  # +1 for background class
        
        # Group by image ID to handle multiple objects per image
        self.image_ids = self.df['Image_ID'].unique()
        self.image_data = {}
        
        for img_id in self.image_ids:
            img_df = self.df[self.df['Image_ID'] == img_id]
            
            # Get image path (same for all rows with this image ID)
            img_path = img_df['ImagePath'].iloc[0]
            
            # Get all objects (bboxes and classes) for this image
            if not is_test:
                boxes = []
                labels = []
                confidences = []
                
                for _, row in img_df.iterrows():
                    # Extract bounding box coordinates
                    if all(coord in row for coord in ['xmin', 'ymin', 'xmax', 'ymax']):
                        box = [
                            float(row['xmin']), 
                            float(row['ymin']), 
                            float(row['xmax']), 
                            float(row['ymax'])
                        ]
                        
                        # Extract class label
                        class_name = row['class']
                        class_idx = self.class_map[class_name] + 1  # +1 because 0 is reserved for background
                        
                        # Extract confidence if available
                        confidence = float(row['confidence']) if 'confidence' in row else 1.0
                        
                        boxes.append(box)
                        labels.append(class_idx)
                        confidences.append(confidence)
                
                # Store all object data for this image
                self.image_data[img_id] = {
                    'path': img_path,
                    'boxes': boxes,
                    'labels': labels,
                    'confidences': confidences
                }
            else:
                # For test set, just store the image path
                self.image_data[img_id] = {
                    'path': img_path
                }
    
    def __len__(self):
        return len(self.image_ids)
    
    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_info = self.image_data[img_id]
        img_path = img_info['path']
        
        # Load image
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            # Return a black image in case of error
            image = Image.new('RGB', (224, 224), color='black')
        
        # Get original image dimensions
        width, height = image.size
        
        # Apply transformations
        if self.transform:
            image = self.transform(image)
        
        # Return image and target (or just image for test set)
        if self.is_test:
            return image, img_id
        else:
            # Convert boxes and labels to tensors
            boxes = torch.FloatTensor(img_info['boxes'])
            labels = torch.LongTensor(img_info['labels'])
            confidences = torch.FloatTensor(img_info['confidences'])
            
            # RCNN target format
            target = {
                'boxes': boxes,
                'labels': labels,
                'confidences': confidences,
                'image_id': torch.tensor([idx])
            }
            
            return image, target

def collate_fn(batch):
    """
    Custom collate function for the DataLoader that handles
    variable-sized targets (different number of objects per image).
    """
    images = []
    targets = []
    
    for img, tgt in batch:
        images.append(img)
        targets.append(tgt)
    
    return images, targets

def prepare_data(df, img_size=224):
    """
    Prepare data for training, validation, and testing.
    
    Args:
        df: DataFrame with image information including bounding boxes
        img_size: Size to resize images to
    
    Returns:
        train_loader, val_loader, test_loader, class_map, classes
    """
    # Create class mapping
    classes = sorted(df['class'].unique())
    class_map = {cls: i for i, cls in enumerate(classes)}
    
    # Define transforms
    train_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Split data into train, validation, and test sets
    # First get unique image IDs
    image_ids = df['Image_ID'].unique()
    train_ids, temp_ids = train_test_split(image_ids, test_size=0.3, random_state=42)
    val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, random_state=42)
    
    # Filter dataframe based on image IDs
    train_df = df[df['Image_ID'].isin(train_ids)]
    val_df = df[df['Image_ID'].isin(val_ids)]
    test_df = df[df['Image_ID'].isin(test_ids)]
    
    # Create datasets
    train_dataset = PlantRCNNDataset(train_df, class_map, transform=train_transform)
    val_dataset = PlantRCNNDataset(val_df, class_map, transform=val_transform)
    test_dataset = PlantRCNNDataset(test_df, class_map, transform=val_transform)
    
    # Create data loaders with multiprocessing settings adjusted
    # Using persistent_workers=True and reducing worker count to prevent multiprocessing issues
    train_loader = DataLoader(
        train_dataset, 
        batch_size=4,  # Smaller batch size for RCNN as it's more memory intensive
        shuffle=True, 
        num_workers=2,
        collate_fn=collate_fn,  # Custom collate function for variable-sized targets
        persistent_workers=True if torch.cuda.is_available() else False,
        pin_memory=torch.cuda.is_available()
    )
    val_loader = DataLoader(
        val_dataset, 
        batch_size=4, 
        shuffle=False, 
        num_workers=2,
        collate_fn=collate_fn,
        persistent_workers=True if torch.cuda.is_available() else False,
        pin_memory=torch.cuda.is_available()
    )
    test_loader = DataLoader(
        test_dataset, 
        batch_size=4, 
        shuffle=False, 
        num_workers=2,
        collate_fn=collate_fn,
        persistent_workers=True if torch.cuda.is_available() else False,
        pin_memory=torch.cuda.is_available()
    )
    
    return train_loader, val_loader, test_loader, class_map, classes

class PlantRCNN(nn.Module):
    def __init__(self, num_classes):
        super(PlantRCNN, self).__init__()
        
        # Load a pre-trained model
        backbone = torchvision.models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        
        # Remove the last two layers (avg pooling and FC)
        backbone = nn.Sequential(*list(backbone.children())[:-2])
        
        # Define feature map channels
        backbone_out_channels = 2048
        
        # RPN anchor generator
        anchor_generator = AnchorGenerator(
            sizes=((32, 64, 128, 256, 512),),
            aspect_ratios=((0.5, 1.0, 2.0),)
        )
        
        # Define RPN head
        rpn_head = torchvision.models.detection.rpn.RPNHead(
            backbone_out_channels, 
            anchor_generator.num_anchors_per_location()[0]
        )
        
        # ROI pooler
        roi_pooler = torchvision.ops.MultiScaleRoIAlign(
            featmap_names=['0'],
            output_size=7,
            sampling_ratio=2
        )
        
        # Box head
        box_head = torchvision.models.detection.faster_rcnn.TwoMLPHead(
            backbone_out_channels * 7 * 7,
            1024
        )
        
        # Box predictor
        box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(
            1024,
            num_classes + 1  # +1 for background class
        )
        
        # Construct the complete model
        self.model = FasterRCNN(
            backbone,
            min_size=224,
            max_size=1333,
            rpn_anchor_generator=anchor_generator,
            rpn_head=rpn_head,
            box_roi_pool=roi_pooler,
            box_head=box_head,
            box_predictor=box_predictor,
            # Add confidence score for each prediction
            box_score_thresh=0.05,
            box_nms_thresh=0.5,
            box_detections_per_img=100
        )
    
    def forward(self, images, targets=None):
        return self.model(images, targets)

def train_model(model, train_loader, val_loader, class_names, num_epochs=20, learning_rate=0.001,
                optimizer_name='sgd', early_stopping=True, patience=5, min_delta=0.001):
    """
    Train the RCNN model for object detection.
    
    Args:
        model: RCNN PyTorch model
        train_loader: Training data loader
        val_loader: Validation data loader
        class_names: List of class names
        num_epochs: Number of training epochs
        learning_rate: Learning rate
        optimizer_name: Name of optimizer to use ('adam', 'sgd', 'adamw', or 'rmsprop')
        early_stopping: Whether to use early stopping
        patience: Number of epochs to wait for improvement before stopping
        min_delta: Minimum change in validation mAP to qualify as improvement
    
    Returns:
        Trained model and training history
    """
    # Add signal handling for graceful termination
    import signal
    import sys
    
    def signal_handler(sig, frame):
        print('Ctrl+C detected, cleaning up...')
        # Clean up multiprocessing resources
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        sys.exit(0)
    
    # Register signal handler
    signal.signal(signal.SIGINT, signal_handler)
    
    # Use CUDA if available
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Set multiprocessing start method if not already set
    import multiprocessing
    try:
        if not multiprocessing.get_start_method():
            multiprocessing.set_start_method('spawn')
    except RuntimeError:
        pass  # Method already set
    
    # Move model to device
    model = model.to(device)
    
    # Create optimizer based on name
    if optimizer_name.lower() == 'adam':
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    elif optimizer_name.lower() == 'sgd':
        optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9, weight_decay=0.0005)
    elif optimizer_name.lower() == 'adamw':
        optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.0005)
    elif optimizer_name.lower() == 'rmsprop':
        optimizer = optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=0.0005)
    else:
        print(f"Warning: Optimizer {optimizer_name} not recognized. Using SGD as default.")
        optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9, weight_decay=0.0005)
    
    # Learning rate scheduler
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)
    
    # Training history
    history = {
        'train_loss': [],
        'val_map': []
    }
    
    best_val_map = 0.0
    best_model_path = 'best_plant_rcnn_model.pth'
    
    # Early stopping variables
    early_stop_counter = 0
    
    # Training loop
    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        
        # Training phase
        model.train()
        train_loss = 0.0
        count = 0
        
        for images, targets in train_loader:
            # Move data to device
            images = [img.to(device) for img in images]
            targets_device = []
            
            for target in targets:
                target_device = {
                    'boxes': target['boxes'].to(device),
                    'labels': target['labels'].to(device),
                    'image_id': target['image_id'].to(device)
                }
                targets_device.append(target_device)
            
            # Zero the parameter gradients
            optimizer.zero_grad()
            
            # Forward pass (returns loss dictionary)
            loss_dict = model(images, targets_device)
            losses = sum(loss for loss in loss_dict.values())
            
            # Backward pass and optimize
            losses.backward()
            optimizer.step()
            
            train_loss += losses.item()
            count += 1
        
        # Calculate training loss for the epoch
        train_loss = train_loss / count if count > 0 else 0
        
        # Validation phase
        model.eval()
        val_map = evaluate_model_rcnn(model, val_loader, class_names, device)
        
        # Update learning rate
        scheduler.step()
        
        # Store history
        history['train_loss'].append(train_loss)
        history['val_map'].append(val_map)
        
        # Print epoch results
        print(f"Train Loss: {train_loss:.4f}, Val mAP: {val_map:.4f}")
        
        # Save best model based on mAP
        if val_map > best_val_map + min_delta:
            best_val_map = val_map
            torch.save(model.state_dict(), best_model_path)
            print(f"Saved best model with mAP: {val_map:.4f}")
            
            # Reset early stopping counter if we found a better model
            early_stop_counter = 0
        else:
            # Increment early stopping counter if we didn't improve enough
            early_stop_counter += 1
            
        # Check if we should stop early
        if early_stopping and early_stop_counter >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            break
    
    # Load best model
    model.load_state_dict(torch.load(best_model_path))
    
    return model, history

def evaluate_model_rcnn(model, data_loader, class_names, device):
    """
    Evaluate the RCNN model using mAP (Mean Average Precision).
    
    Args:
        model: RCNN PyTorch model
        data_loader: Validation or test data loader
        class_names: List of class names
        device: Device to run the evaluation on
    
    Returns:
        mAP score
    """
    model.eval()
    
    # Lists to store predictions and ground truth for mAP calculation
    all_predictions = []
    all_ground_truth = []
    
    with torch.no_grad():
        for images, targets in data_loader:
            images = [img.to(device) for img in images]
            
            # Get model predictions
            predictions = model(images)
            
            # Process each image's predictions and ground truth
            for i, pred in enumerate(predictions):
                # Get ground truth for this image
                gt_boxes = targets[i]['boxes'].to(device)
                gt_labels = targets[i]['labels'].to(device)
                
                # Get predictions
                pred_boxes = pred['boxes']
                pred_scores = pred['scores']
                pred_labels = pred['labels']
                
                # Store predictions and ground truth for this image
                all_predictions.append({
                    'boxes': pred_boxes,
                    'scores': pred_scores,
                    'labels': pred_labels
                })
                
                all_ground_truth.append({
                    'boxes': gt_boxes,
                    'labels': gt_labels
                })
    
    # Calculate mAP
    map_score = calculate_map_rcnn(all_predictions, all_ground_truth, len(class_names) + 1)
    
    return map_score

def calculate_map_rcnn(predictions, ground_truth, num_classes, iou_threshold=0.5):
    """
    Calculate mAP for object detection based on predictions and ground truth.
    
    Args:
        predictions: List of prediction dictionaries
        ground_truth: List of ground truth dictionaries
        num_classes: Number of classes (including background)
        iou_threshold: IoU threshold for considering a prediction correct
    
    Returns:
        mAP score
    """
    # Initialize variables for mAP calculation
    aps = []
    
    # Calculate AP for each class (skipping the background class)
    for class_id in range(1, num_classes):  # Start from 1 to skip background
        # Lists to store true positives, false positives, and false negatives
        all_detections = []
        all_ground_truths = []
        
        # Collect all detections and ground truths for this class
        for i in range(len(predictions)):
            pred = predictions[i]
            gt = ground_truth[i]
            
            # Get pred_boxes and scores for this class
            class_mask = pred['labels'] == class_id
            pred_boxes = pred['boxes'][class_mask]
            pred_scores = pred['scores'][class_mask]
            
            # Create detections for mAP calculation (box + score)
            for j in range(len(pred_boxes)):
                all_detections.append({
                    'image_id': i,
                    'bbox': pred_boxes[j],
                    'score': pred_scores[j]
                })
            
            # Get ground truth boxes for this class
            gt_mask = gt['labels'] == class_id
            gt_boxes = gt['boxes'][gt_mask]
            
            for j in range(len(gt_boxes)):
                all_ground_truths.append({
                    'image_id': i,
                    'bbox': gt_boxes[j],
                    'used': False  # To mark ground truths that have been matched
                })
        
        # If no ground truths for this class, skip
        if len(all_ground_truths) == 0:
            continue
        
        # Sort detections by confidence score
        all_detections = sorted(all_detections, key=lambda x: x['score'], reverse=True)
        
        # Variables for precision-recall calculation
        true_positives = np.zeros(len(all_detections))
        false_positives = np.zeros(len(all_detections))
        
        # Match detections to ground truths
        for i, detection in enumerate(all_detections):
            img_id = detection['image_id']
            pred_box = detection['bbox']
            
            # Get ground truths for this image
            img_gt = [gt for gt in all_ground_truths if gt['image_id'] == img_id]
            
            # If no ground truths for this image, mark as false positive
            if len(img_gt) == 0:
                false_positives[i] = 1
                continue
            
            # Find best matching ground truth
            max_iou = -1
            max_gt_idx = -1
            
            for j, gt in enumerate(img_gt):
                # Skip if already used
                if gt['used']:
                    continue
                
                # Calculate IoU
                gt_box = gt['bbox']
                iou = box_iou(pred_box.unsqueeze(0), gt_box.unsqueeze(0))
                iou = iou.item()
                
                if iou > max_iou:
                    max_iou = iou
                    max_gt_idx = j
            
            # If IoU above threshold, mark as true positive
            if max_iou >= iou_threshold:
                # Mark the ground truth as used
                img_gt[max_gt_idx]['used'] = True
                true_positives[i] = 1
            else:
                false_positives[i] = 1
        
        # Compute cumulative precision and recall
        cum_true_positives = np.cumsum(true_positives)
        cum_false_positives = np.cumsum(false_positives)
        
        recalls = cum_true_positives / len(all_ground_truths)
        precisions = cum_true_positives / (cum_true_positives + cum_false_positives)
        
        # Compute average precision (AP) using 11-point interpolation
        ap = 0
        for t in np.arange(0, 1.1, 0.1):
            if np.sum(recalls >= t) == 0:
                p = 0
            else:
                p = np.max(precisions[recalls >= t])
            ap += p / 11
        
        aps.append(ap)
    
    # Calculate mAP
    mAP = np.mean(aps) if len(aps) > 0 else 0.0
    
    return mAP

def predict_image(model, image_path, class_names, device=None, conf_threshold=0.5):
    """
    Predict bounding boxes and classes for a single image.
    
    Args:
        model: Trained RCNN PyTorch model
        image_path: Path to the image
        class_names: List of class names
        device: Device to run the prediction on
        conf_threshold: Confidence threshold for predictions
    
    Returns:
        Image with bounding boxes, list of predictions
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Set model to evaluation mode
    model.eval()
    
    # Load and transform image
    image = Image.open(image_path).convert('RGB')
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    transformed_image = transform(image).unsqueeze(0).to(device)
    
    # Make prediction
    with torch.no_grad():
        prediction = model([transformed_image[0]])[0]
    
    # Get prediction details
    boxes = prediction['boxes'].cpu().numpy()
    scores = prediction['scores'].cpu().numpy()
    labels = prediction['labels'].cpu().numpy()
    
    # Filter predictions based on confidence threshold
    mask = scores >= conf_threshold
    boxes = boxes[mask]
    scores = scores[mask]
    labels = labels[mask]
    
    # Convert image to numpy array for visualization
    image_np = np.array(image)
    
    # Draw bounding boxes
    for box, score, label in zip(boxes, scores, labels):
        # Skip background class (label 0)
        if label == 0:
            continue
        
        # Get class name (subtract 1 because class_map adds 1 to indices)
        class_name = class_names[label - 1] if label - 1 < len(class_names) else "Unknown"
        
        # Draw box
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(image_np, (x1, y1), (x2, y2), (0, 255, 0), 2)
        
        # Draw label
        label_text = f"{class_name}: {score:.2f}"
        cv2.putText(image_np, label_text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    
    # Create list of predictions
    predictions = []
    for box, score, label in zip(boxes, scores, labels):
        # Skip background class
        if label == 0:
            continue
        
        class_name = class_names[label - 1] if label - 1 < len(class_names) else "Unknown"
        predictions.append({
            'class': class_name,
            'confidence': float(score),
            'bbox': {
                'xmin': float(box[0]),
                'ymin': float(box[1]),
                'xmax': float(box[2]),
                'ymax': float(box[3])
            }
        })
    
    return image_np, predictions

# Example usage
def main():
    # Load data
    # Assuming df has columns: Image_ID, ImagePath, class, xmin, ymin, xmax, ymax, confidence
    df = pd.read_csv('your_data.csv')
    
    # Prepare data
    train_loader, val_loader, test_loader, class_map, class_names = prepare_data(df)
    
    # Create model
    model = PlantRCNN(len(class_names))
    
    # Train model
    trained_model, history = train_model(
        model, 
        train_loader, 
        val_loader, 
        class_names, 
        num_epochs=20, 
        learning_rate=0.001, 
        optimizer_name='sgd'
    )
    
    # Evaluate model
    evaluate_model_rcnn(trained_model, test_loader, class_names, 
                        torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    
    # Predict on a sample image
    sample_image_path = 'path/to/sample/image.jpg'
    result_image, predictions = predict_image(trained_model, sample_image_path, class_names)
    
    # Display result
    plt.figure(figsize=(12, 8))
    plt.imshow(result_image)
    plt.title('Object Detection Results')
    plt.axis('off')
    plt.show()
    
    # Print predictions
    for pred in predictions:
        print(f"Class: {pred['class']}, Confidence: {pred['confidence']:.2f}")
        print(f"Bounding Box: {pred['bbox']}")
        print("---")

if __name__ == "__main__":
    main()